# Google Colab GPU — FIDES full-session showcase

Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy lần lượt từng cell. Notebook build native FIDES component từ các Git submodule đã pin trong repository; không cài OpenFHE CPU wheel.

## 1. Kiểm tra GPU

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("he-sdk-fides hiện hỗ trợ đúng Python 3.12")

compute_capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).splitlines()[0].strip()

os.environ["FIDESLIB_ARCH"] = compute_capability.replace(".", "") + "-real"
os.environ["HE_REPO"] = "/content/k3s-demo-app"
os.environ["HE_LOCAL_PREFIX"] = "/content/he-fides-local"
os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "2"

print("FIDESLIB_ARCH:", os.environ["FIDESLIB_ARCH"])

## 2. Cài công cụ build

Chỉ cài compiler/build tools. OpenFHE và FIDESlib sẽ được build từ đúng source đã pin trong submodule.

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends build-essential git libomp-dev ninja-build patch python3-dev
%pip install --upgrade pip
%pip install 'cmake==3.31.6' 'scikit-build-core>=0.10,<1' 'pybind11>=2.13,<4'

## 3. Clone repository và toàn bộ submodule

In [ ]:
%%bash
set -euo pipefail

HE_REPO="${HE_REPO:-/content/k3s-demo-app}"

if [[ ! -d "${HE_REPO}/.git" ]]; then
  git clone --recurse-submodules \
    https://gitlab.com/nhatcao99uetwork/k3s-demo-app.git \
    "${HE_REPO}"
else
  git -C "${HE_REPO}" pull --ff-only origin main
  git -C "${HE_REPO}" submodule update --init --recursive
fi

git -C "${HE_REPO}" rev-parse --short HEAD
git -C "${HE_REPO}" submodule status --recursive

## 4. Áp dụng OpenFHE patch của FIDESlib

FIDESlib phải dùng patched OpenFHE nằm trong submodule; không cài hoặc import stock `openfhe` trong runtime GPU này.

In [ ]:
%%bash
set -euo pipefail

HE_REPO="${HE_REPO:-/content/k3s-demo-app}"

OPENFHE_SOURCE="${HE_REPO}/gpu/third_party/FIDESlib/deps/openfhe-src"
FIDES_PATCH="${HE_REPO}/gpu/third_party/FIDESlib/deps/fideslib-ref-1.5.1.1.patch"

if patch --dry-run --forward --batch --directory="${OPENFHE_SOURCE}" --strip=1 < "${FIDES_PATCH}" >/dev/null; then
  patch --forward --batch --directory="${OPENFHE_SOURCE}" --strip=1 < "${FIDES_PATCH}"
elif patch --dry-run --reverse --batch --directory="${OPENFHE_SOURCE}" --strip=1 < "${FIDES_PATCH}" >/dev/null; then
  echo "FIDES OpenFHE patch is already applied"
else
  echo "Patch does not match the pinned OpenFHE source" >&2
  exit 1
fi

## 5. Build patched OpenFHE

Hai cell build native có thể mất nhiều phút. Parallelism được giữ ở mức `2` để giảm nguy cơ Colab hết RAM.

In [ ]:
%%bash
set -euo pipefail

HE_REPO="${HE_REPO:-/content/k3s-demo-app}"
HE_LOCAL_PREFIX="${HE_LOCAL_PREFIX:-/content/he-fides-local}"
CMAKE_BUILD_PARALLEL_LEVEL="${CMAKE_BUILD_PARALLEL_LEVEL:-2}"

SOURCE="${HE_REPO}/gpu/third_party/FIDESlib/deps/openfhe-src"
BUILD="/content/he-fides-build/openfhe"
INSTALL="${HE_LOCAL_PREFIX}/openfhe"

cmake -S "${SOURCE}" -B "${BUILD}" -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_INSTALL_PREFIX="${INSTALL}" \
  -DCMAKE_POSITION_INDEPENDENT_CODE=ON \
  -DBUILD_STATIC=ON \
  -DBUILD_SHARED=OFF \
  -DBUILD_UNITTESTS=OFF \
  -DBUILD_EXAMPLES=OFF \
  -DBUILD_BENCHMARKS=OFF \
  -DGIT_SUBMOD_AUTO=OFF \
  -DWITH_NATIVEOPT=OFF

cmake --build "${BUILD}" --parallel "${CMAKE_BUILD_PARALLEL_LEVEL}"
cmake --install "${BUILD}"

## 6. Build và install FIDESlib

In [ ]:
%%bash
set -euo pipefail

HE_REPO="${HE_REPO:-/content/k3s-demo-app}"
HE_LOCAL_PREFIX="${HE_LOCAL_PREFIX:-/content/he-fides-local}"
CMAKE_BUILD_PARALLEL_LEVEL="${CMAKE_BUILD_PARALLEL_LEVEL:-2}"
FIDESLIB_ARCH="${FIDESLIB_ARCH:-75-real}"

SOURCE="${HE_REPO}/gpu/third_party/FIDESlib"
BUILD="/content/he-fides-build/fideslib"
INSTALL="${HE_LOCAL_PREFIX}/fideslib"

cmake -S "${SOURCE}" -B "${BUILD}" -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_POSITION_INDEPENDENT_CODE=ON \
  -DFIDESLIB_ARCH="${FIDESLIB_ARCH}" \
  -DFIDESLIB_INSTALL_OPENFHE=OFF \
  -DFIDESLIB_COMPILE_TESTS=OFF \
  -DFIDESLIB_COMPILE_BENCHMARKS=OFF \
  -DFIDESLIB_INSTALL_PREFIX="${INSTALL}" \
  -DOPENFHE_INSTALL_PREFIX="${HE_LOCAL_PREFIX}/openfhe"

cmake --build "${BUILD}" --target install --parallel "${CMAKE_BUILD_PARALLEL_LEVEL}"

## 7. Cài core SDK và native GPU binding

Core được cài editable không kèm CPU dependency. Binding được build trong Colab và liên kết với FIDESlib/patched OpenFHE vừa cài.

In [ ]:
%%bash
set -euo pipefail

HE_REPO="${HE_REPO:-/content/k3s-demo-app}"
HE_LOCAL_PREFIX="${HE_LOCAL_PREFIX:-/content/he-fides-local}"

python -m pip uninstall -y openfhe he-sdk-fides he_looming_sdk 2>/dev/null || true
python -m pip install --no-deps -e "${HE_REPO}"

export CMAKE_PREFIX_PATH="${HE_LOCAL_PREFIX}/fideslib;${HE_LOCAL_PREFIX}/openfhe"
export CMAKE_ARGS="-DCMAKE_PREFIX_PATH=${CMAKE_PREFIX_PATH}"
python -m pip install \
  --no-deps \
  --no-build-isolation \
  --force-reinstall \
  "${HE_REPO}/gpu/he_sdk_fides"

## 8. Tạo GPU session

In [ ]:
from he_sdk import HESession, __version__
import he_sdk_fides

session = HESession.create(device="gpu")

left_values = [1.0, 2.0, 3.0, 4.0]
right_values = [10.0, 20.0, 30.0, 40.0]

print("he_sdk version:", __version__)
print("he_sdk_fides version:", he_sdk_fides.__version__)
print("selected backend:", session.capabilities.backend)
print("capabilities:", session.capabilities)

## 9. Encrypt và decrypt trên FIDES GPU

In [ ]:
left_ct = session.encrypt(left_values)
right_ct = session.encrypt(right_values)

print("left input:", left_values)
print("encrypted left:", left_ct)
print("decrypted left:", session.decrypt(left_ct))

## 10. Add và subtract

In [ ]:
add_ct = session.add(left_ct, right_ct)
subtract_ct = session.subtract(left_ct, right_ct)

print("add expected:", [11.0, 22.0, 33.0, 44.0])
print("add decrypted:", session.decrypt(add_ct))
print("subtract expected:", [-9.0, -18.0, -27.0, -36.0])
print("subtract decrypted:", session.decrypt(subtract_ct))

## 11. Multiply và square

In [ ]:
multiply_ct = session.multiply(left_ct, right_ct)
square_ct = session.square(left_ct)

print("multiply expected:", [10.0, 40.0, 90.0, 160.0])
print("multiply decrypted:", session.decrypt(multiply_ct))
print("square expected:", [1.0, 4.0, 9.0, 16.0])
print("square decrypted:", session.decrypt(square_ct))

## 12. Sum, mean và variance

In [ ]:
sum_ct = session.sum(left_ct)
mean_ct = session.mean(left_ct)
variance_ct = session.variance(left_ct)

print("sum expected:", 10.0)
print("sum decrypted:", session.decrypt(sum_ct))
print("mean expected:", 2.5)
print("mean decrypted:", session.decrypt(mean_ct))
print("variance expected:", 1.25)
print("variance decrypted:", session.decrypt(variance_ct))

## 13. Đóng session

Local FIDES backend hiện chưa hỗ trợ workspace serialization và recipient/PRE.

In [ ]:
session.close()
print("GPU session closed")